In [20]:
import json
import re
from datasets import Dataset
from sklearn.model_selection import train_test_split
# from seqeval.metrics import f1_score, precision_score, recall_score
# import gc
# import numpy as np
# import pandas as pd
from pathlib import Path


In [2]:
!pip install datasets

  Using cached pyarrow-25.0.1-cp314-cp314-win_amd64.whl.metadata (3.0 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached multiprocess-0.70.19-py314-none-any.whl.metadata (7.2 kB)
  Using cached aiohappyeyeballs-2.7.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached frozenlist-1.8.0-cp314-cp314-win_amd64.whl.metadata (21 kB)
   ---------------------------------------- 0.0/604.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/604.4 kB ? eta -:--:--
   ---------------------------------------- 604.4/604.4 kB 2.9 MB/s  0:00:00
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
Using cached multiprocess-0.70.19-py314-none-any.whl (160 kB)
Using cached aiohappyeyeballs-2.7.1-py3-none-any.whl (15 kB)
Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
Using cached attrs-26.1.0-py3-none-any.whl (67 kB)
Using cached froze


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
!pip install scikit-learn

  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached narwhals-2.26.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.4 MB 4.6 MB/s eta 0:00:02
   --- ------------------------------------ 0.8/8.4 MB 4.6 MB/s eta 0:00:02
   ------- -------------------------------- 1.6/8.4 MB 2.4 MB/s eta 0:00:03
   -------- ------------------------------- 1.8/8.4 MB 2.4 MB/s eta 0:00:03
   ---------- ----------------------------- 2.1/8.4 MB 2.3 MB/s eta 0:00:03
   ------------- -------------------------- 2.9/8.4 MB 2.2 MB/s eta 0:00:03
   ---------------- ----------------------- 3.4/8.4 MB 2.3 MB/s eta 0:00:03
   ---------------- ----------------------- 3.4/8.4 MB 2.3 MB/s e


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
MAX_WORDS = 250
OVERLAP = 40

In [24]:
dataset_path = r"C:\Users\Админ\PycharmProjects\NER\data\ner_dataset\Entity Recognition in Resumes.json"

In [11]:
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]", re.UNICODE)

In [12]:
def tokenize_with_offsets(text):
    """Токенизирует текст и возвращает токены + их (start, end) позиции в исходной строке."""
    tokens = []
    spans = []
    for match in TOKEN_PATTERN.finditer(text):
        tokens.append(match.group())
        spans.append((match.start(), match.end()))
    return tokens, spans


In [15]:
def resolve_overlaps(annotations):
    """
    Убирает пересекающиеся аннотации.
    Приоритет отдаётся более длинному span.

    Формат annotation:
    [
        {
            'label': ['Skills'],
            'points': [
                {
                    'start': 1295,
                    'end': 1621,
                    'text': '...'
                }
            ]
        },
        ...
    ]
    """

    # Сначала превращаем структуру в удобный формат:
    # (start, end, label, text)
    spans = []

    for annotation in annotations:
        if not annotation.get('label'):
            continue
        if not annotation.get('points'):
            continue
            
       

        # Если points пустой — пропускаем аннотацию
       
        label = annotation['label'][0]
        point = annotation['points'][0]

        start = point['start']
        end = point['end']
        text = point['text']

        spans.append((start, end, label, text))

    # Сначала самые длинные spans
    spans.sort(key=lambda x: x[1] - x[0], reverse=True)

    resolved = []
    occupied = []

    for start, end, label, text in spans:

        # Проверяем пересечение с уже выбранными spans
        overlap = any(
            not (end <= o_start or start >= o_end)
            for o_start, o_end in occupied
        )

        if not overlap:
            resolved.append({
                'label': [label],
                'points': [{
                    'start': start,
                    'end': end,
                    'text': text
                }]
            })

            occupied.append((start, end))

    # Возвращаем в порядке появления в тексте
    resolved.sort(
        key=lambda x: x['points'][0]['start']
    )

    return resolved

In [16]:
def convert_to_bio(text, annotations):
    """text + annotations -> tokens + BIO-теги."""

    tokens, spans = tokenize_with_offsets(text)
    tags = ["O"] * len(tokens)

    clean_annotations = resolve_overlaps(annotations)

    for annotation in clean_annotations:

        entity_type = annotation['label'][0].upper()

        point = annotation['points'][0]
        start = point['start']
        end = point['end']+1

        is_first_token = True

        for i, (tok_start, tok_end) in enumerate(spans):

            # Токен полностью находится внутри аннотации
            if tok_start >= start and tok_end <= end:

                tags[i] = (
                    "B-" if is_first_token else "I-"
                ) + entity_type

                is_first_token = False

    return tokens, tags

In [17]:
def convert_dataset(df):
    documents = []
    file_num = 1

    for _, row in df.iterrows():

        text = row.get("content", "")
        annotations = row.get("annotation", [])

        if not text or not annotations:
            continue

        tokens, tags = convert_to_bio(text, annotations)

        documents.append({
            "file": f"cv_{file_num:02d}",
            "tokens": tokens,
            "tags": tags
        })

        file_num += 1

    return documents

In [27]:
# import json
import pandas as pd
# from pathlib import Path

# Путь к локальному датасету
dataset_path = Path(dataset_path)

# Загружаем датасет
dataset = []

with open(dataset_path, "r", encoding="utf-8") as f:
    for line in f:
        dataset.append(json.loads(line))

# Для удобного анализа через Pandas
df = pd.DataFrame(dataset)

print(f"Успешно загружено {len(df)} резюме")
print(df.head(2))

Успешно загружено 220 резюме
                                             content  \
0  Abhishek Jha\nApplication Development Associat...   
1  Afreen Jamadar\nActive member of IIIT Committe...   

                                          annotation extras  
0  [{'label': ['Skills'], 'points': [{'start': 12...   None  
1  [{'label': ['Email Address'], 'points': [{'sta...   None  


In [28]:
the_dataset = convert_dataset(df)


In [29]:


def chunk_document(tokens, tags):
    if len(tokens) <= MAX_WORDS:
        return [(tokens, tags)]
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + MAX_WORDS, len(tokens))
        chunks.append((tokens[start:end], tags[start:end]))
        if end >= len(tokens):
            break
        start += MAX_WORDS - OVERLAP
    return chunks



In [32]:
all_chunks = []
for data in the_dataset:
    all_chunks.extend(chunk_document(data["tokens"], data["tags"]))

print(f"Чанков: {len(all_chunks)}")

Чанков: 788


In [34]:
#from sklearn.model_selection import train_test_split
train_chunks, val_chunks = train_test_split(
    all_chunks, test_size=0.15, random_state=42
)

In [35]:
all_tags = sorted(set(tag for data in the_dataset for tag in data["tags"]))
tag2id = {tag: i for i, tag in enumerate(all_tags)}
id2tag = {i: tag for tag, i in tag2id.items()}

In [36]:
id2tag

{0: 'B-COLLEGE NAME',
 1: 'B-COMPANIES WORKED AT',
 2: 'B-DEGREE',
 3: 'B-DESIGNATION',
 4: 'B-EMAIL ADDRESS',
 5: 'B-GRADUATION YEAR',
 6: 'B-LOCATION',
 7: 'B-NAME',
 8: 'B-SKILLS',
 9: 'B-UNKNOWN',
 10: 'B-YEARS OF EXPERIENCE',
 11: 'I-COLLEGE NAME',
 12: 'I-COMPANIES WORKED AT',
 13: 'I-DEGREE',
 14: 'I-DESIGNATION',
 15: 'I-EMAIL ADDRESS',
 16: 'I-GRADUATION YEAR',
 17: 'I-LOCATION',
 18: 'I-NAME',
 19: 'I-SKILLS',
 20: 'I-UNKNOWN',
 21: 'I-YEARS OF EXPERIENCE',
 22: 'O'}

In [19]:
#from datasets import Dataset

def chunks_to_dataset(chunks):
    return Dataset.from_dict({
        "tokens": [c[0] for c in chunks],
        "ner_tags": [[tag2id[t] for t in c[1]] for c in chunks],
    })

ds_train = chunks_to_dataset(train_chunks)
ds_val = chunks_to_dataset(val_chunks)
print(f"Train: {len(ds_train)}, Val: {len(ds_val)}")

Train: 669, Val: 119


In [24]:
def tokenize_and_align(examples, tokenizer):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=512,
        padding=False,
    )
    all_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word_id:
                label_ids.append(labels[word_id])
            else:
                label_ids.append(-100)
            prev_word_id = word_id
        all_labels.append(label_ids)
    tokenized["labels"] = all_labels
    return tokenized

In [61]:

for token, label in zip(tokens, labels):
    tag = id2tag[label] if label != -100 else "-100"
    print(f"{token:30} {tag}")

<s>                            -100
▁Sta                           O
ck                             -100
▁                              O
,                              -100
▁S                             O
SH                             -100
▁                              O
,                              -100
▁S                             O
TP                             -100
▁device                        O
▁driver                        O
s                              -100
▁                              O
,                              -100
▁B                             O
GP                             -100
▁protocol                      O
▁                              O
,                              -100
▁M                             O
LT                             -100
▁                              O
,                              -100
▁LA                            O
CP                             -100
▁                              O
,                              -100
▁

In [6]:
!pip install -U pip setuptools wheel
!pip install --no-build-isolation seqeval==1.2.1

  Using cached pip-26.2.1-py3-none-any.whl.metadata (4.6 kB)
  Using cached wheel-0.48.0-py3-none-any.whl.metadata (2.3 kB)
Using cached pip-26.2.1-py3-none-any.whl (1.8 MB)
Using cached wheel-0.48.0-py3-none-any.whl (33 kB)



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: To modify pip, please run the following command:
C:\Users\Админ\PycharmProjects\NER\.venv\Scripts\python.exe -m pip install -U pip setuptools wheel


  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Discarding https://files.pythonhosted.org/packages/c4/47/f85f522f1f2532ca650474088e4024a9f52d524379bff058eaadf53cb663/seqeval-1.2.1.tar.gz (from https://pypi.org/simple/seqeval/): Requested seqeval==1.2.1 from https://files.pythonhosted.org/packages/c4/47/f85f522f1f2532ca650474088e4024a9f52d524379bff058eaadf53cb663/seqeval-1.2.1.tar.gz has inconsistent version: expected '1.2.1', but metadata has '0.0.0'


ERROR: Could not find a version that satisfies the requirement seqeval==1.2.1 (from versions: 0.0.1, 0.0.2, 0.0.3, 0.0.4, 0.0.5, 0.0.6, 0.0.7, 0.0.8, 0.0.9, 0.0.10, 0.0.11, 0.0.12, 0.0.13, 0.0.15, 0.0.16, 0.0.17, 0.0.18, 0.0.19, 1.0.0, 1.1.0, 1.1.1, 1.2.0, 1.2.1, 1.2.2)

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for seqeval==1.2.1


In [7]:
!pip install git+https://github.com/chakki-works/seqeval.git

  Cloning https://github.com/chakki-works/seqeval.git to C:\Users\Админ\AppData\Local\Temp\pip-req-build-rktityr_


  ERROR: Error [WinError 2] Не удается найти указанный файл while executing command git version

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Cannot find command 'git' - do you have 'git' installed and in your PATH?


In [50]:
# import gc
# import numpy as np

from seqeval.metrics import f1_score


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_labels, true_preds = [], []

    for pred_seq, label_seq in zip(predictions, labels):
        pred_tags, label_tags = [], []

        for p, l in zip(pred_seq, label_seq):
            if l != -100:
                pred_tags.append(id2tag[int(p)])
                label_tags.append(id2tag[int(l)])

        true_preds.append(pred_tags)
        true_labels.append(label_tags)

    return {"f1": f1_score(true_labels, true_preds)}


In [52]:
def evaluate_detailed(trainer, tokenized_val):
    """Возвращает подробные метрики по каждой сущности."""
    predictions, labels, _ = trainer.predict(tokenized_val)
    predictions = np.argmax(predictions, axis=-1)

    true_labels, true_preds = [], []
    for pred_seq, label_seq in zip(predictions, labels):
        pred_tags, label_tags = [], []
        for p, l in zip(pred_seq, label_seq):
            if l != -100:
                pred_tags.append(id2tag[int(p)])
                label_tags.append(id2tag[int(l)])
        true_preds.append(pred_tags)
        true_labels.append(label_tags)

    report = classification_report(true_labels, true_preds, output_dict=True)
    f1_overall = f1_score(true_labels, true_preds)
    p_overall = precision_score(true_labels, true_preds)
    r_overall = recall_score(true_labels, true_preds)

    return {
        "report": report,
        "f1": f1_overall,
        "precision": p_overall,
        "recall": r_overall,
        "report_text": classification_report(true_labels, true_preds),
    }

In [27]:
# import time
# import gc
# import torch

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    DataCollatorForTokenClassification,
    Trainer,
)

# ============================================================
# МОДЕЛЬ
# ============================================================

model_name = "XLM-RoBERTa-base"
model_id = "xlm-roberta-base"

print("=" * 70)
print(f"  {model_name} ({model_id})")
print("=" * 70)



  XLM-RoBERTa-base (xlm-roberta-base)


In [54]:
try:

    # ========================================================
    # 1. Токенизатор
    # ========================================================

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    print("\n[1/5] Tokenizer loaded")

    # ========================================================
    # 2. Токенизация train / validation
    # ========================================================

    tok_fn = lambda examples: tokenize_and_align(
        examples,
        tokenizer
    )

    tokenized_train = ds_train.map(
        tok_fn,
        batched=True,
        remove_columns=ds_train.column_names
    )

    tokenized_val = ds_val.map(
        tok_fn,
        batched=True,
        remove_columns=ds_val.column_names
    )

    print("[2/5] Dataset tokenized")

    # ========================================================
    # 3. Модель
    # ========================================================

    model = AutoModelForTokenClassification.from_pretrained(
        model_id,
        num_labels=len(tag2id),
        id2label=id2tag,
        label2id=tag2id,
    )

    n_params = model.num_parameters() / 1e6

    print(f"[3/5] Model loaded")
    print(f"      Parameters: {n_params:.0f}M")

    # ========================================================
    # 4. Training arguments
    # ========================================================

    batch_size = 8

    training_args = TrainingArguments(
        output_dir=f"./ner-{model_name}",

        num_train_epochs=20,

        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,

        learning_rate=5e-5,
        weight_decay=0.01,
        warmup_ratio=0.1,

        eval_strategy="epoch",
        save_strategy="epoch",

        logging_steps=20,

        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,

        save_total_limit=2,

        fp16=torch.cuda.is_available(),

        report_to="none",
        seed=42,
    )

    data_collator = DataCollatorForTokenClassification(
        tokenizer=tokenizer
    )

    # ========================================================
    # 5. Trainer
    # ========================================================

    trainer = Trainer(
        model=model,
        args=training_args,

        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,

        processing_class=tokenizer,

        data_collator=data_collator,

        compute_metrics=compute_metrics,
    )

    print("[4/5] Trainer ready")

    # ========================================================
    # ОБУЧЕНИЕ
    # ========================================================

    print("\n" + "=" * 70)
    print("  TRAINING")
    print("=" * 70)

    start_time = time.time()

    trainer.train()

    train_time = time.time() - start_time

    # ========================================================
    # ОЦЕНКА
    # ========================================================

    print("\n" + "=" * 70)
    print("  EVALUATION")
    print("=" * 70)

    metrics = evaluate_detailed(
        trainer,
        tokenized_val
    )

    metrics["train_time"] = train_time
    metrics["params_m"] = n_params
    metrics["model_id"] = model_id

    results = {
        model_name: metrics
    }

    print(
        f"\nF1: {metrics['f1']:.4f}"
        f"  |  Time: {train_time:.0f}s"
        f"  |  Params: {n_params:.0f}M"
    )

    print(metrics["report_text"])

except Exception as e:

    print(f"\nОШИБКА: {e}")

    results = {
        model_name: {
            "f1": 0,
            "error": str(e)
        }
    }

finally:

    # ========================================================
    # Очистка памяти
    # ========================================================

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "=" * 70)
print("  ОБУЧЕНИЕ ЗАВЕРШЕНО")
print("=" * 70)


[1/5] Tokenizer loaded


Map:   0%|          | 0/669 [00:00<?, ? examples/s]

Map:   0%|          | 0/119 [00:00<?, ? examples/s]

[2/5] Dataset tokenized


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


  EVALUATION



ОШИБКА: name 'classification_report' is not defined

  ОБУЧЕНИЕ ЗАВЕРШЕНО
